# Second-Order Equations and Oscillations

Newton's law gives accelerations, so mechanics is full of second-order equations $y'' = g(t, y, y')$. Writing $v = y'$ turns each into a first-order system, which is how the panel and every solver handle them. This notebook covers the linear oscillator, damping, forcing and resonance, and the nonlinear pendulum.

In [ ]:
Pollis.packages("OrdinaryDiffEq", "Plots")

In [ ]:
using OrdinaryDiffEq, LinearAlgebra, Plots

## Damping regimes

The damped oscillator $y'' + 2\zeta\omega y' + \omega^2 y = 0$ has natural frequency $\omega$ and damping ratio $\zeta$. The characteristic roots are $-\zeta\omega \pm \omega\sqrt{\zeta^2 - 1}$, which are also the eigenvalues of the companion matrix of the first-order system.

In [ ]:
w = 2.0
companion(zeta) = [0.0 1.0; -w^2 -2zeta * w]
[(zeta = z, regime = z < 1 ? "underdamped" : z == 1 ? "critically damped" : "overdamped", roots = eigvals(companion(z))) for z in (0.2, 1.0, 2.0)]

In [ ]:
p = plot(xlabel="t", ylabel="y", title="Released from y = 1 at rest")
for z in (0.2, 1.0, 2.0)
    plot!(p, solve(ODEProblem((u, p, t) -> companion(z) * u, [1.0, 0.0], (0.0, 10.0)), Tsit5()), idxs=1, lw=2, label="zeta = $z")
end
p

The underdamped solution is $e^{-\zeta\omega t}\left(\cos\omega_d t + \frac{\zeta\omega}{\omega_d}\sin\omega_d t\right)$ with $\omega_d = \omega\sqrt{1 - \zeta^2}$:

In [ ]:
z = 0.2; wd = w * sqrt(1 - z^2)
exact(t) = exp(-z * w * t) * (cos(wd * t) + z * w / wd * sin(wd * t))
sol = solve(ODEProblem((u, p, t) -> companion(z) * u, [1.0, 0.0], (0.0, 10.0)), Tsit5(); reltol=1e-10, abstol=1e-12)
maximum(abs(sol(t)[1] - exact(t)) for t in 0:0.01:10)

## Forcing and resonance

Driving the oscillator with $\cos(\Omega t)$ gives, once the transient has died out, an oscillation of amplitude

$$\frac{1}{\sqrt{(\omega^2 - \Omega^2)^2 + (2\zeta\omega\Omega)^2}},$$

which peaks near $\Omega = \omega$: **resonance**. Here each forcing frequency is simulated to $t = 200$, and the amplitude is measured over the last 50 time units.

In [ ]:
z = 0.1
forced(W) = (u, p, t) -> [u[2], -2z * w * u[2] - w^2 * u[1] + cos(W * t)]
function amplitude(W)
    s = solve(ODEProblem(forced(W), [0.0, 0.0], (0.0, 200.0)), Tsit5(); reltol=1e-8, abstol=1e-10)
    return maximum(abs(s(t)[1]) for t in range(150, 200, length=2000))
end
theory(W) = 1 / sqrt((w^2 - W^2)^2 + (2z * w * W)^2)
Ws = range(0.5, 4.0, length=60)
plot(Ws, theory.(Ws), lw=2, label="formula", xlabel="forcing frequency", ylabel="steady-state amplitude", title="Resonance near the natural frequency $w")
scatter!(Ws, amplitude.(Ws), ms=3, label="simulated")

## The nonlinear pendulum

$\theta'' = -(g/\ell)\sin\theta$. For small angles $\sin\theta \approx \theta$ and the period is $2\pi\sqrt{\ell/g}$ whatever the amplitude. For large swings the period grows. The exact period is $4\sqrt{\ell/g}\,K(\sin(\theta_0/2))$, where $K$ is the complete elliptic integral. $K$ can be computed with Gauss's arithmetic-geometric mean: $K(k) = \pi / (2\,\mathrm{agm}(1, \sqrt{1 - k^2}))$.

In [ ]:
gl = 9.81   # g / length, for a 1-metre pendulum
pend(u, p, t) = [u[2], -gl * sin(u[1])]
function agm(a, b)
    for _ in 1:30
        a, b = (a + b) / 2, sqrt(a * b)
    end
    return a
end
exact_period(theta0) = 2pi / sqrt(gl) / agm(1.0, cos(theta0 / 2))
function measured_period(theta0)
    s = solve(ODEProblem(pend, [theta0, 0.0], (0.0, 20.0)), Tsit5(); reltol=1e-10, abstol=1e-12)
    ts = range(0, 20, length=20001); th = [s(t)[1] for t in ts]
    ups = [ts[i] for i in 2:length(ts) if th[i - 1] < 0 <= th[i]]   # upward zero crossings
    return (ups[end] - ups[1]) / (length(ups) - 1)
end
[(amplitude = a, small_angle_period = 2pi / sqrt(gl), exact_period = exact_period(a), measured = measured_period(a)) for a in (0.1, 0.5, 1.0, 2.0, 2.5, 3.0)]

## The phase plane and energy

Without damping, the energy $E = \tfrac12\theta'^2 - (g/\ell)\cos\theta$ is conserved, so trajectories follow its level curves. Closed curves are swings; open curves at the top and bottom are full rotations; the curve between them is the **separatrix**. Damping, as in the panel's default pendulum, makes trajectories spiral down to rest.

In [ ]:
E(th, v) = v^2 / 2 - gl * cos(th)
contour(range(-2pi, 2pi, length=300), range(-8, 8, length=200), E, levels=20, c=:grays, xlabel="angle", ylabel="angular velocity", title="Pendulum: energy levels and a damped swing", colorbar=false)
damped = solve(ODEProblem((u, p, t) -> [u[2], -0.5u[2] - gl * sin(u[1])], [2.5, 0.0], (0.0, 20.0)), Tsit5())
plot!(damped, idxs=(1, 2), lw=2, c=:red, label="damping 0.5")

In [ ]:
s = solve(ODEProblem(pend, [2.5, 0.0], (0.0, 100.0)), Tsit5(); reltol=1e-8, abstol=1e-10)
energies = [E(u[1], u[2]) for u in s.u]
(energy_at_start = energies[1], largest_drift = maximum(abs.(energies .- energies[1])))

## Exercises

1. For $\zeta = 0.1$, at which forcing frequency is the amplitude largest? Compare with $\omega\sqrt{1 - 2\zeta^2}$.
2. An RLC circuit obeys $Lq'' + Rq' + q/C = V(t)$. Find $\omega$ and $\zeta$ in terms of $L$, $R$, $C$ and simulate $L = 1$, $R = 0.5$, $C = 0.25$ with a unit step voltage.
3. Start the pendulum at $\theta_0 = 0$ with $\theta'_0 = 6.5$. Does it swing or rotate? Use the energy to predict the answer first.
4. Solve the undamped pendulum to $t = 10^4$ with Tsit5 at `reltol=1e-6`, then with `KahanLi8()` on a `SecondOrderODEProblem` with `dt=0.01`. Compare the energy drift.